# Clase 2 — Actividad práctica: regresión y generalización
**Machine Learning · Especialización en Analítica de Datos**
Dr. Oscar Loyola Valenzuela

**Objetivo.** Construir modelos de regresión lineal para estimar el valor mediano de las viviendas por distrito, evaluarlos con datos no utilizados en el entrenamiento y controlar el sobreajuste mediante regularización.

**Conjunto de datos.** California Housing (Pace y Barry, 1997), en la versión distribuida por Géron (2022), que incluye la variable categórica `ocean_proximity` y valores faltantes en `total_bedrooms`.

> Ejecute las celdas en orden. Las celdas marcadas como **Ejercicio** requieren completar código o responder por escrito.

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

## 1. Revisión exploratoria y preparación de variables

In [ ]:
print("Faltantes por columna:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDistritos con el valor topado:", (df["median_house_value"] >= 500000).sum())
print("\nCategorías de ocean_proximity:")
print(df["ocean_proximity"].value_counts())

In [ ]:
# Unidades: objetivo en miles de USD; ingreso en decenas de miles de USD
df["valor"] = df["median_house_value"] / 1000
df["ingreso"] = df["median_income"]

# Variables relativas al tamaño del distrito
df["habitaciones_por_hogar"] = df["total_rooms"] / df["households"]
df["dormitorios_por_habitacion"] = df["total_bedrooms"] / df["total_rooms"]
df["personas_por_hogar"] = df["population"] / df["households"]

NUM = ["longitude", "latitude", "housing_median_age", "ingreso",
       "habitaciones_por_hogar", "dormitorios_por_habitacion", "personas_por_hogar", "households"]
CAT = ["ocean_proximity"]

X, y = df[NUM + CAT], df["valor"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print("Entrenamiento:", len(X_tr), "| Prueba:", len(X_te))

In [ ]:
def evaluar(modelo, nombre, cols=None):
    """Devuelve MAE, RMSE y R2 en entrenamiento y prueba."""
    A, B = (X_tr, X_te) if cols is None else (X_tr[cols], X_te[cols])
    filas = []
    for conjunto, XX, yy in [("Entrenamiento", A, y_tr), ("Prueba", B, y_te)]:
        p = modelo.predict(XX)
        filas.append({"modelo": nombre, "conjunto": conjunto,
                      "MAE": mean_absolute_error(yy, p),
                      "RMSE": mean_squared_error(yy, p) ** 0.5,
                      "R2": r2_score(yy, p)})
    return pd.DataFrame(filas).round(3)

## 2. Regresión lineal simple

In [ ]:
simple = LinearRegression().fit(X_tr[["ingreso"]], y_tr)
print(f"beta_0 = {simple.intercept_:.2f}   beta_1 = {simple.coef_[0]:.2f}")
evaluar(simple, "Simple", ["ingreso"])

In [ ]:
muestra = X_tr.assign(valor=y_tr).sample(2500, random_state=1)
gx = np.linspace(0.5, 15, 50)
plt.figure(figsize=(7, 4.5))
plt.scatter(muestra["ingreso"], muestra["valor"], s=6, alpha=0.3)
plt.plot(gx, simple.intercept_ + simple.coef_[0] * gx, color="black", lw=2)
plt.xlabel("Ingreso mediano (decenas de miles de USD)")
plt.ylabel("Valor mediano (miles de USD)")
plt.show()

### Ejercicio 1
1. Interprete $\beta_0$ y $\beta_1$ en las unidades del problema. ¿Tiene sentido práctico el intercepto?
2. ¿Qué patrón se observa en la parte superior del gráfico y cómo afecta a la recta?

## 3. Regresión lineal múltiple con `Pipeline`

In [ ]:
def preprocesador(grado=1):
    pasos = [SimpleImputer(strategy="median"), StandardScaler()]
    if grado > 1:
        pasos.append(PolynomialFeatures(grado, include_bias=False))
    return make_column_transformer(
        (make_pipeline(*pasos), NUM),
        (OneHotEncoder(drop="first", handle_unknown="ignore"), CAT))

multiple = make_pipeline(preprocesador(), LinearRegression()).fit(X_tr, y_tr)
resultados = pd.concat([evaluar(simple, "Simple", ["ingreso"]), evaluar(multiple, "Múltiple")])
resultados

In [ ]:
nombres = [n.split("__")[1] for n in multiple[0].get_feature_names_out()]
coef = pd.Series(multiple[-1].coef_, index=nombres).sort_values()
coef.plot.barh(figsize=(7, 5))
plt.xlabel("Coeficiente (miles de USD por desviación estándar)")
plt.show()

In [ ]:
p_te = multiple.predict(X_te)
plt.figure(figsize=(6, 5))
plt.scatter(y_te, p_te, s=5, alpha=0.3)
plt.plot([0, 520], [0, 520], "k--")
plt.xlabel("Valor real (miles de USD)"); plt.ylabel("Valor predicho (miles de USD)")
plt.show()
print("Predicciones negativas en prueba:", (p_te < 0).sum())

### Ejercicio 2
1. ¿Qué variables tienen mayor efecto según los coeficientes estandarizados?
2. ¿Por qué no es correcto afirmar que «aumentar la latitud disminuye el precio»?

## 4. Ejercicio 3 — Efecto de los valores topados
Repita el modelo múltiple **excluyendo** los distritos con `median_house_value >= 500000`. Recuerde realizar una nueva partición sobre el conjunto filtrado.

In [ ]:
# Complete aquí
filtro = df["median_house_value"] < 500000
Xf, yf = df.loc[filtro, NUM + CAT], df.loc[filtro, "valor"]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2, random_state=42)

m_f = make_pipeline(preprocesador(), LinearRegression()).fit(Xf_tr, yf_tr)
p = m_f.predict(Xf_te)
print(f"Sin topados -> MAE: {mean_absolute_error(yf_te, p):.1f}  "
      f"RMSE: {mean_squared_error(yf_te, p) ** 0.5:.1f}  R2: {r2_score(yf_te, p):.3f}")

**Pregunta.** Las métricas cambian, pero ¿son comparables con las del modelo original? ¿Para qué distritos sería válido el nuevo modelo?

## 5. Complejidad y regularización
### 5.1 Términos polinomiales

In [ ]:
tabla = []
for grado in [1, 2, 3]:
    m = make_pipeline(preprocesador(grado), LinearRegression()).fit(X_tr, y_tr)
    tabla.append(evaluar(m, f"Grado {grado}"))
pd.concat(tabla)

### Ejercicio 4 — Regularización Ridge
Ajuste el modelo de grado 3 con `Ridge` para $\lambda \in \{1, 100, 1000\}$ (parámetro `alpha`) y compare con la tabla anterior.

In [ ]:
# Complete aquí
tabla_ridge = []
for lam in [1, 100, 1000]:
    m = make_pipeline(preprocesador(3), Ridge(alpha=lam)).fit(X_tr, y_tr)
    tabla_ridge.append(evaluar(m, f"Grado 3, Ridge lambda={lam}"))
pd.concat(tabla_ridge)

**Preguntas.**
1. ¿Qué valor de $\lambda$ ofrece el mejor equilibrio entre entrenamiento y prueba?
2. En este ejercicio, $\lambda$ se eligió observando el conjunto de prueba. ¿Por qué es una práctica incorrecta y qué alternativa propone?

## 6. Discusión final
1. ¿Qué métrica comunicaría a un gerente inmobiliario y por qué?
2. Un error medio de 51 000 USD, ¿es aceptable? ¿De qué depende la respuesta?

---
**Referencias**
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly.
- Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55–67.
- Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters*, 33(3), 291–297.